# Part D: Full RAG Pipeline & Adversarial Evaluation
**Student:** Dabone Abdoul Latif  
**Index:** 10012200015

This notebook integrates the full RAG pipeline and conducts critical adversarial testing to compare its performance against a pure LLM.

### 1. Setup and Initialization

In [ ]:
import json
import faiss
import os
import numpy as np
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from groq import Groq
from dotenv import load_dotenv

load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

with open('chunks/csv_chunks.json', 'r') as f: csv_chunks = json.load(f)
with open('chunks/pdf_chunks.json', 'r') as f: pdf_chunks = json.load(f)
all_chunks = csv_chunks + pdf_chunks

model = SentenceTransformer('all-MiniLM-L6-v2')
index = faiss.read_index('indexes/rag_index.faiss')

texts = [c['text'] for c in all_chunks]
bm25 = BM25Okapi([t.lower().split() for t in texts])

### 2. The RAG Pipeline Function
Includes logging at each stage as required by the project specifications.

In [ ]:
def rag_pipeline(query, k=3, verbose=True):
    if verbose: print(f"[LOG] Stage 1: Retrieving chunks for: '{query}'")
    
    # Hybrid Search
    query_vec = model.encode([query]).astype('float32')
    distances, indices = index.search(query_vec, k * 2)
    bm25_scores = bm25.get_scores(query.lower().split())
    max_bm25 = max(bm25_scores) if max(bm25_scores) > 0 else 1
    
    combined = []
    for i, chunk in enumerate(all_chunks):
        v_score = 0
        for rank, idx in enumerate(indices[0]):
            if idx == i: v_score = 1 / (1 + distances[0][rank])
        score = (0.5 * v_score) + (0.5 * (bm25_scores[i] / max_bm25))
        if score > 0: combined.append({'chunk': chunk, 'score': score})
    
    results = sorted(combined, key=lambda x: x['score'], reverse=True)[:k]
    
    if verbose:
        for i, res in enumerate(results):
            print(f"      - Chunk {i+1} | Score: {res['score']:.4f} | Source: {res['chunk']['source']}")
    
    # Context & Prompt
    context = "".join([f"\n[Source: {r['chunk']['source']}] {r['chunk']['text']}\n" for r in results])
    
    final_prompt = f"""Answer based ONLY on context. If not found, say you don't know.\nDocuments: {context}\nQuery: {query}\nAnswer:"""
    
    if verbose: 
        print("[LOG] Stage 3: Constructed Prompt sent to LLM")
        print("[LOG] Stage 4: Querying LLM...")
    
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": final_prompt}],
        temperature=0
    )
    return response.choices[0].message.content

### 3. Adversarial Testing & Comparison
Here we compare the RAG system against a Pure LLM using misleading queries.

In [ ]:
def pure_llm_response(query):
    res = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": query}],
        temperature=0
    )
    return res.choices[0].message.content

trick_query = "How many votes did the NPP get in the 2025 budget?"
print(f"Adversarial Query: {trick_query}\n")

print("--- PURE LLM RESPONSE (Hallucination Risk) ---")
print(pure_llm_response(trick_query))

print("\n--- RAG PIPELINE RESPONSE (Grounded) ---")
print(rag_pipeline(trick_query))